# 01 · Data collection: from Tiki's search API to a clean table

**Goal:** collect the products listed for a keyword, understand the raw data, and turn it into a clean table that is ready for analysis.

This notebook walks through, step by step, what the pipeline (`run_crawl.py`) does automatically:

1. Check that collecting the data is acceptable
2. Understand what the API returns
3. Build the raw table
4. Clean it (and measure what each rule removes)
5. Build the sample dataset used in `02_price_analysis.ipynb`

In [1]:
import json
import sys
import urllib.robotparser
from pathlib import Path

import pandas as pd

# make the project packages importable from the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import config
from analysis.cleaning import clean_products, is_relevant, matches_keyword, parse_sold
from analysis.pricing import add_analysis_columns
from crawler import tiki

pd.set_option("display.max_colwidth", 60)
KEYWORD = "chuột gaming"
config.REQUEST_DELAY = 2  # this notebook sends more requests than one web search: go slower

## 1. Is it OK to collect this data?

Before writing a crawler I check three things:

- **robots.txt** — does the website ask bots not to visit this path?
- **Access** — the endpoint is the public JSON API that tiki.vn itself calls from the browser (found in DevTools → Network). No login, no cookie.
- **Volume** — at most 5 pages per keyword, a 1-second pause between requests, results re-used for 30 minutes. The data is used for personal learning only.

In [2]:
robots = urllib.robotparser.RobotFileParser("https://tiki.vn/robots.txt")
robots.read()
print("Allowed to fetch /api/v2/products:", robots.can_fetch("*", "https://tiki.vn/api/v2/products?q=test"))

Allowed to fetch /api/v2/products: True


`robotparser` only compares the beginning of each path: it does not understand the `*` wildcards that Tiki's robots.txt uses (for example `Disallow: /*airdrop*`). So I also read the file myself: the wildcard rules block spam search terms, and the other rules block account, cart and checkout pages and `/api/v2/me/`. None of them covers `/api/v2/products`.

## 2. What does the API return?

One request = one page of search results. The response also tells us how many pages exist (`paging`).

In [3]:
data = tiki.get_page(KEYWORD, page=1)  # same function the crawler uses (waits and retries if rate limited)

print("Paging:", {k: data["paging"][k] for k in ["total", "per_page", "last_page"]})
print("Products on this page:", len(data["data"]))
print("Fields per product:", len(data["data"][0]))

Paging: {'total': 1891, 'per_page': 40, 'last_page': 48}
Products on this page: 40
Fields per product: 46


In [4]:
# one raw product, only the fields we keep
item = data["data"][0]
fields = ["id", "name", "brand_name", "seller_name", "primary_category_name", "price",
          "original_price", "quantity_sold", "rating_average", "review_count", "url_path"]
print(json.dumps({f: item.get(f) for f in fields}, ensure_ascii=False, indent=2))

{
  "id": 56196502,
  "name": "Chuột Gaming Logitech G102 Gen 2 Lightsync - Hàng Chính Hãng",
  "brand_name": "Logitech",
  "seller_name": "Tiki Trading",
  "primary_category_name": "Chuột Chơi Game",
  "price": 471900,
  "original_price": 471900,
  "quantity_sold": {
    "text": "Đã bán 4757",
    "value": 4757
  },
  "rating_average": 4.8,
  "review_count": 1392,
  "url_path": "chuot-gaming-logitech-g102-gen-2-lightsync-hang-chinh-hang-p56196502.html?spid=73161623"
}


Each product has ~45 fields; 12 of them answer our questions. The mapping used in `crawler/tiki.py`:

| API field | Column | Note |
|---|---|---|
| `id` | `product_id` | unique id on Tiki |
| `name` | `name` | |
| `brand_name` | `brand` | "OEM" / "No Brand" → empty during cleaning |
| `seller_name` | `seller` | |
| `primary_category_name` | `category` | used to filter and to compare similar products |
| `price` | `price` | selling price, VND |
| `original_price` | `original_price` | price before discount |
| `quantity_sold.value` | `sold` | nested object, missing for new products |
| `rating_average` | `rating` | 0 means *not rated yet* |
| `review_count` | `review_count` | |
| `thumbnail_url` | `image_url` | |
| `url_path` | `product_url` | `https://tiki.vn/` + path |

> **Bug found in v1:** the first version read the product link from the *advertisement* field. Today no product in the response has that field, so every Tiki row was dropped during cleaning and the crawler silently returned **0 products**. Version 2 builds the link from `url_path`.

In [5]:
print("Products with an 'advertisement' field:", sum("advertisement" in p for p in data["data"]), "/", len(data["data"]))
print("Products with a 'url_path' field:      ", sum(bool(p.get("url_path")) for p in data["data"]), "/", len(data["data"]))

Products with an 'advertisement' field: 0 / 40
Products with a 'url_path' field:       40 / 40


## 3. The raw table

`tiki.crawl()` downloads up to 5 pages (200 products) and keeps the mapped fields.

In [6]:
raw = tiki.crawl(KEYWORD)
print(raw.shape)
raw.head(3)

(200, 13)


,source,product_id,name,brand,seller,category,price,original_price,sold,rating,review_count,image_url,product_url
0,Tiki,56196502,Chuột Gaming Logitech G102 Gen 2 Lightsync - Hàng Chính ...,Logitech,Tiki Trading,Chuột Chơi Game,471900,471900,4757,4.8,1392,https://salt.tikicdn.com/cache/280x280/ts/product/f8/39/...,https://tiki.vn/chuot-gaming-logitech-g102-gen-2-lightsy...
1,Tiki,276611072,Chuột Gaming Có Dây Newmen Game G10+ Hàng Chính Hãng,Newmen,VINSEC,Chuột Chơi Game,199000,199000,18,5.0,1,https://salt.tikicdn.com/cache/280x280/ts/product/87/a6/...,https://tiki.vn/chuot-gaming-co-day-newmen-game-g10-hang...
2,Tiki,195854829,Chuột Gaming E-DRA EM6102w - Hàng Chính Hãng,E-DRA,VINSEC,Chuột Chơi Game,243000,243000,6,0.0,0,https://salt.tikicdn.com/cache/280x280/ts/product/0e/ff/...,https://tiki.vn/chuot-gaming-e-dra-em6102w-p195854829.ht...


In [7]:
# data quality: share of missing values per column
quality = pd.DataFrame({
    "missing %": (raw.isna().mean() * 100).round(1),
    "zero %": ((raw == 0).mean() * 100).round(1),
})
quality[(quality["missing %"] > 0) | (quality["zero %"] > 0)]

,missing %,zero %
sold,0.0,40.0
rating,0.0,63.5
review_count,0.0,63.5


Observations:

- `rating` is **0** for many products: that means *no rating yet*, not *zero stars*. Averaging it as 0 would pull the average rating down (a bug in v1, where ratings were also stored as text).
- `sold` is 0 for new products — that is a real 0 and is kept.
- `brand` contains placeholder values such as "OEM".

## 4. Cleaning, step by step

The cleaning rules live in `analysis/cleaning.py`. Here they are applied one by one to see how many rows each rule removes.

In [8]:
funnel = {"raw rows": len(raw)}

df = raw.dropna(subset=["name", "product_url", "price"])
df = df[df["price"] > 0]
funnel["valid name, link and price"] = len(df)

df = df.drop_duplicates(subset=["source", "product_id"])
funnel["no duplicates"] = len(df)

relevant = is_relevant(df, KEYWORD)
funnel["relevant to the keyword"] = int(relevant.sum())

pd.Series(funnel, name="rows").to_frame()

,rows
raw rows,200
"valid name, link and price",200
no duplicates,200
relevant to the keyword,173


### The relevance rule

Search results also contain loosely related items. A product is kept when:

- its name contains **every word** of the keyword (accent-insensitive: "chuot gaming" matches "Chuột Gaming"), **or**
- it belongs to the **main category** of those exact matches.

The second rule fixes a weakness of v1, which used only the first rule and dropped many real gaming mice whose name says *"chơi game"* instead of *"gaming"*.

In [9]:
exact = df["name"].apply(matches_keyword, keyword=KEYWORD)
rescued = relevant & ~exact

print("Exact name match:          ", int(exact.sum()))
print("Kept by the category rule: ", int(rescued.sum()))
print("Removed:                   ", int((~relevant).sum()))
df.loc[rescued, ["name", "category"]].head(5)

Exact name match:           113
Kept by the category rule:  60
Removed:                    27


,name,category
38,Chuột không dây chơi game HXSJ X801 thiết kế độc lạ Led ...,Chuột Chơi Game
39,Chuột Chơi Game EM602 Led - Hàng chính hãng,Chuột Chơi Game
71,Chuột Game Chuyên Dụng Đèn Led - Màu ngẫu nhiên,Chuột Chơi Game
73,"Chuột Game Có Dây iCore GM03 - LED 7 Màu, DPI 7200, Giá ...",Chuột Chơi Game
74,Chuột không dây chơi game X801 thiết kế độc lạ Led RGB đ...,Chuột Chơi Game


In [10]:
# what the rule removes
df.loc[~relevant, ["name", "category"]].head(5)

,name,category
100,Chuột Có Dây T-WOLF G520 Led RGB – Hàng Chính Hãng,Chuột Văn Phòng Có Dây
107,Chuột vi tính có dây cổng USB R8 1637 LED màu trắng đen-...,Bàn Di Chuột - Miếng Lót Chuột
116,Chuột Có Dây T-Wolf V1 LED USB – Hàng Chính Hãng,Chuột Văn Phòng Có Dây
136,Chuột quang chơi game không dây 2.4G cao cấp - miếng lót...,Bàn Di Chuột - Miếng Lót Chuột
140,Chuột quang chơi game pc latop tivi không dây 2.4G bình ...,Bàn Di Chuột - Miếng Lót Chuột


The rule is not perfect: a *mouse + mouse pad* bundle that the seller listed under **mouse pads** is removed, and so are a few gaming-style mice listed under **office mice**. Category labels come from sellers, so this is a precision/recall trade-off. Keeping the `category` column also lets the dashboard analyse those groups separately.

### Sold counts written as text

Lazada writes sold counts as text (`"1.2K sold"`). The v1 parser replaced `k` with `e3` and then removed every non-digit character — including the `e` — so `"1.2K sold"` became **1** instead of 1 200. The new `parse_sold()`:

In [11]:
examples = ["1.2K sold", "2.7K sold", "206 sold", "3,5k", "Đã bán 1.234", 4757, None]
pd.DataFrame({"input": examples, "parse_sold()": [parse_sold(x) for x in examples]})

,input,parse_sold()
0,1.2K sold,1200
1,2.7K sold,2700
2,206 sold,206
3,"3,5k",3500
4,Đã bán 1.234,1234
5,4757,4757
6,None,0


### The clean table

In [12]:
clean = clean_products(raw, KEYWORD)
print(clean.shape)
clean.dtypes

(173, 14)


source                str
product_id            str
name                  str
brand                 str
seller                str
category              str
price               int64
original_price      int64
discount_pct        int64
sold                int64
rating            float64
review_count        int64
image_url             str
product_url           str
dtype: object

In [13]:
clean[["price", "original_price", "discount_pct", "sold", "rating", "review_count"]].describe().round(1)

,price,original_price,discount_pct,sold,rating,review_count
count,173.0,173.0,173.0,173.0,65.0,173.0
mean,700111.2,736531.2,1.9,74.2,4.5,19.5
std,892761.4,1013851.9,6.9,465.9,0.9,121.7
min,22093.0,22093.0,0.0,0.0,1.0,0.0
25%,199000.0,199000.0,0.0,0.0,4.5,0.0
50%,355000.0,355000.0,0.0,1.0,5.0,0.0
75%,690000.0,749000.0,0.0,9.0,5.0,1.0
max,5359000.0,7149000.0,50.0,4757.0,5.0,1392.0


## 5. Build the sample dataset

Three keywords from different product types, collected slowly (2 s between pages, 30 s between keywords). Each one goes through the same pipeline (crawl → clean → analysis columns) and is saved to `data/sample/tiki_sample.csv`, so `02_price_analysis.ipynb` can be re-run any time with exactly the same data.

Re-collecting would overwrite that file, and every number in notebook 02 and in the README would change (prices move every day). So by default this cell only loads the saved sample; set `REBUILD_SAMPLE = True` to collect a new one.

In [14]:
import time

REBUILD_SAMPLE = False  # True: crawl again and overwrite data/sample/tiki_sample.csv
KEYWORDS = ["chuột gaming", "tai nghe bluetooth", "nồi chiên không dầu"]
sample_path = ROOT / "data" / "sample" / "tiki_sample.csv"

if REBUILD_SAMPLE:
    crawled_at = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    frames = []
    for keyword in KEYWORDS:
        if keyword == KEYWORD:
            keyword_raw = raw  # already downloaded in section 3
        else:
            time.sleep(30)     # be polite: a long pause between two keywords
            keyword_raw = tiki.crawl(keyword)
        products = add_analysis_columns(clean_products(keyword_raw, keyword))
        products.insert(0, "keyword", keyword)
        products["crawled_at"] = crawled_at
        frames.append(products)
        print(f"{keyword:<22} {len(products):>4} products")

    sample = pd.concat(frames, ignore_index=True)
    sample.to_csv(sample_path, index=False, encoding="utf-8-sig")
    print("Saved", sample.shape, "->", sample_path.relative_to(ROOT))
else:
    sample = pd.read_csv(sample_path)
    print("Using the saved sample:", sample.shape, "collected at", sample["crawled_at"].iloc[0])
    print(sample.groupby("keyword", sort=False).size().to_string())

Using the saved sample: (567, 18) collected at 2026-10-24 10:27
keyword
chuột gaming           173
tai nghe bluetooth     197
nồi chiên không dầu    197


**Next:** `02_price_analysis.ipynb` analyses this dataset.